# Financial Inclusion in the Philippines
### From Financial Access to Digital Financial Engagement

**Data:** Global Findex 2025  
**Tools:** Python · Pandas · NumPy · Plotly · SciPy · Google BigQuery  
**Methods:** Survey-Weighted Analysis · Segmentation · Gap Analysis · Chi-Square · Cramér's V

---

## Executive Summary

This project examines financial inclusion in the Philippines across three connected dimensions:

**Financial Access → Digital Activation → Financial Behavior**

Using the Philippines subset of the Global Findex 2025 microdata, the analysis identifies demographic gaps in financial account ownership, measures digital-payment activation among account owners, compares saving and borrowing behavior across financial-engagement levels, and statistically validates the major descriptive findings.

### Headline Findings

- **Education** produced the largest observed account-ownership gap at **48.53 percentage points**.
- **80.37% of account owners use digital payments**, leaving a **19.63% digital activation gap**.
- **Age and internet usage** were the strongest observed differentiators of digital-payment adoption among account owners.
- Financial engagement differentiated **saving behavior more strongly than borrowing behavior**.

> Supporting code is intentionally collapsed where possible so the notebook reads as an analytical case study. Expand a code cell to inspect the implementation.

## Project Objectives

### Primary Objective
Identify demographic and behavioral characteristics associated with financial exclusion and digital financial adoption among Filipino adults.

### Research Questions

**RQ1 — Financial Access**  
Which demographic groups in the Philippines show the largest gaps in financial account ownership?

**RQ2 — Digital Activation**  
Among financial account owners, which groups show the largest differences in digital-payment adoption?

**RQ3 — Financial Behavior**  
How do saving and borrowing behaviors differ across financial-engagement levels?

# Phase 1 — Dataset Overview & Validation

This phase establishes the analytical sample, validates respondent-level uniqueness, reviews missing values, and retains only variables required by the research questions.

In [ ]:
# Import libraries

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)

In [ ]:
# Load Global Findex data from Google BigQuery

from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

client = bigquery.Client(
    project="sprint-2-final-day-project"
)

query = '''
SELECT *
FROM `sprint-2-final-day-project.findex_dataset.findex_microdata_2025_labelled_update112425`
'''

data = client.query(query).to_dataframe()

print("Full dataset:", data.shape)

## Dataset Overview

The source file contains approximately **144,090 respondents and 199 variables**.  
The analysis is limited to respondents whose economy is recorded as **Philippines**.

In [ ]:
# Create Philippines analysis sample

ph = data[
    data["economy"] == "Philippines"
].copy()

selected_columns = [
    "wpid_random",
    "wgt",

    # Demographics
    "female",
    "age",
    "educ",
    "inc_q",
    "emp_in",
    "urbanicity",

    # Financial access
    "account",
    "account_fin",
    "account_mob",
    "dig_account",

    # Digital behavior
    "anydigpayment",
    "internet_use",

    # Financial behavior
    "saved",
    "borrowed",

    # Payment products retained for reference
    "fin2",
    "fin10"
]

df = ph[selected_columns].copy()

print("Philippines respondents:", len(ph))
print("Analysis dataset shape:", df.shape)

In [ ]:
# Respondent-level validation

validation_summary = pd.DataFrame({
    "Metric": [
        "Total Respondents",
        "Unique Respondent IDs",
        "Duplicate Respondent IDs",
        "Missing Survey Weights"
    ],
    "Value": [
        len(df),
        df["wpid_random"].nunique(),
        df["wpid_random"].duplicated().sum(),
        df["wgt"].isna().sum()
    ]
})

validation_summary

In [ ]:
# Missing-value assessment for analysis variables

missing_values = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2)
}).sort_values(
    "Missing %",
    ascending=False
)

missing_values

## Data Quality Summary

- The Philippines subset contains **1,000 respondents**.
- `wpid_random` is used to validate respondent-level uniqueness.
- Survey weights are retained for population-level descriptive estimates.
- Missing values are reviewed before variables are used in analysis.
- Raw source codes are preserved alongside readable derived variables for traceability.

# Phase 2 — Data Preparation & Feature Engineering

Raw survey variables are converted into readable analytical categories. Two custom segments are also created:

- **Financial Engagement:** Financially Excluded, Basic Access, Digitally Active
- **Financial Behavior:** Saver Only, Borrower Only, Saver & Borrower, Neither

These are analyst-created segments and are not official Global Findex classifications.

In [ ]:
# Demographic labels

df["Gender"] = df["female"].map({
    1: "Female",
    2: "Male"
})

df["Education"] = df["educ"].map({
    1: "Primary or Less",
    2: "Secondary",
    3: "Tertiary"
})

df["Income_Quintile"] = df["inc_q"].map({
    1: "Q1 - Poorest",
    2: "Q2 - Lower Middle",
    3: "Q3 - Middle",
    4: "Q4 - Upper Middle",
    5: "Q5 - Richest"
})

df["Employment"] = df["emp_in"].map({
    1: "In Workforce",
    2: "Out of Workforce"
})

# Official variable represents rural residence.
df["Residence"] = df["urbanicity"].map({
    1: "Rural",
    2: "Urban"
})

df["Age_Group"] = pd.cut(
    df["age"],
    bins=[15, 25, 35, 45, 55, 65, np.inf],
    labels=["15-24", "25-34", "35-44", "45-54", "55-64", "65+"],
    right=False
)

In [ ]:
# Readable financial indicators

df["Account_Status"] = df["account"].map({
    1: "Account Owner",
    0: "No Account"
})

df["Digital_Account_Status"] = df["dig_account"].map({
    1: "Digital Account",
    0: "No Digital Account"
})

df["Digital_Payment_Status"] = df["anydigpayment"].map({
    1: "Digital Payment User",
    0: "Non-Digital Payment User"
})

df["Internet_Status"] = df["internet_use"].map({
    1: "Internet User",
    0: "Non-Internet User"
})

df["Saving_Status"] = df["saved"].map({
    1: "Saved",
    0: "Did Not Save"
})

df["Borrowing_Status"] = df["borrowed"].map({
    1: "Borrowed",
    0: "Did Not Borrow"
})

In [ ]:
# Analytical segmentation

behavior_conditions = [
    (df["saved"] == 1) & (df["borrowed"] == 0),
    (df["saved"] == 0) & (df["borrowed"] == 1),
    (df["saved"] == 1) & (df["borrowed"] == 1),
    (df["saved"] == 0) & (df["borrowed"] == 0)
]

df["Financial_Behavior"] = np.select(
    behavior_conditions,
    [
        "Saver Only",
        "Borrower Only",
        "Saver & Borrower",
        "Neither"
    ],
    default="Unknown"
)

engagement_conditions = [
    df["account"] == 0,
    (df["account"] == 1) & (df["anydigpayment"] == 0),
    (df["account"] == 1) & (df["anydigpayment"] == 1)
]

df["Financial_Engagement"] = np.select(
    engagement_conditions,
    [
        "Financially Excluded",
        "Basic Access",
        "Digitally Active"
    ],
    default="Unknown"
)

In [ ]:
# Transformation validation

transformation_validation = pd.DataFrame({
    "Metric": [
        "Total Respondents",
        "Unique Respondents",
        "Missing Account Status",
        "Missing Digital Payment Status",
        "Missing Survey Weight",
        "Unknown Financial Behavior",
        "Unknown Financial Engagement"
    ],
    "Value": [
        len(df),
        df["wpid_random"].nunique(),
        df["Account_Status"].isna().sum(),
        df["Digital_Payment_Status"].isna().sum(),
        df["wgt"].isna().sum(),
        (df["Financial_Behavior"] == "Unknown").sum(),
        (df["Financial_Engagement"] == "Unknown").sum()
    ]
})

transformation_validation

## Preparation Summary

The final analysis dataset preserves original source fields while adding readable categories and purpose-built analytical segments. No respondents are intentionally removed during feature engineering.

## Reusable Analysis Functions

Survey weights are used for the main population-level descriptive percentages.

In [ ]:
def weighted_percentage(data, condition, weight="wgt"):
    """Return the weighted percentage of rows meeting a condition."""
    return (
        data.loc[condition, weight].sum()
        / data[weight].sum()
        * 100
    )


def weighted_group_percentage(
    data,
    group_column,
    outcome_column,
    outcome_value=1,
    weight="wgt"
):
    """Calculate a weighted outcome percentage for each group."""

    results = []

    for group in data[group_column].dropna().unique():
        group_data = data[
            data[group_column] == group
        ]

        rate = weighted_percentage(
            group_data,
            group_data[outcome_column] == outcome_value,
            weight
        )

        results.append({
            group_column: group,
            "Respondents": len(group_data),
            "Weighted_Percentage": round(rate, 2)
        })

    return pd.DataFrame(results)

# Phase 3 — RQ1: Financial Access

## Research Question
**Which demographic groups in the Philippines show the largest gaps in financial account ownership?**

The analysis compares weighted account-ownership rates by education, income, age, gender, employment, and residence.

In [ ]:
#@title Financial Inclusion Snapshot

kpi_summary = pd.DataFrame({
    "Indicator": [
        "Financial Account",
        "Digitally Enabled Account",
        "Digital Payment User",
        "Internet User",
        "Saved Money",
        "Borrowed Money"
    ],
    "Weighted_Percentage": [
        weighted_percentage(df, df["account"] == 1),
        weighted_percentage(df, df["dig_account"] == 1),
        weighted_percentage(df, df["anydigpayment"] == 1),
        weighted_percentage(df, df["internet_use"] == 1),
        weighted_percentage(df, df["saved"] == 1),
        weighted_percentage(df, df["borrowed"] == 1)
    ]
})

kpi_summary["Weighted_Percentage"] = (
    kpi_summary["Weighted_Percentage"].round(2)
)

kpi_summary

In [ ]:
# Account ownership by demographic group

demographic_factors = {
    "Gender": "Gender",
    "Education": "Education",
    "Income": "Income_Quintile",
    "Age": "Age_Group",
    "Employment": "Employment",
    "Residence": "Residence"
}

account_results = {}

for factor, column in demographic_factors.items():
    account_results[factor] = weighted_group_percentage(
        df,
        column,
        "account"
    )

gap_rows = []

for factor, table in account_results.items():
    column = demographic_factors[factor]

    lowest = table.loc[
        table["Weighted_Percentage"].idxmin()
    ]

    highest = table.loc[
        table["Weighted_Percentage"].idxmax()
    ]

    gap_rows.append({
        "Factor": factor,
        "Lowest_Group": lowest[column],
        "Lowest_Rate": lowest["Weighted_Percentage"],
        "Highest_Group": highest[column],
        "Highest_Rate": highest["Weighted_Percentage"],
        "Gap_pp": round(
            highest["Weighted_Percentage"]
            - lowest["Weighted_Percentage"],
            2
        )
    })

gap_summary = (
    pd.DataFrame(gap_rows)
    .sort_values("Gap_pp", ascending=False)
    .reset_index(drop=True)
)

gap_summary

In [ ]:
#@title Largest Account Ownership Gaps

plot_data = (
    gap_summary
    .head(3)
    .sort_values("Gap_pp")
)

fig = px.bar(
    plot_data,
    x="Gap_pp",
    y="Factor",
    orientation="h",
    text="Gap_pp",
    custom_data=[
        "Lowest_Group",
        "Lowest_Rate",
        "Highest_Group",
        "Highest_Rate"
    ]
)

fig.update_traces(
    texttemplate="<b>%{text:.1f} pp</b>",
    textposition="outside",
    cliponaxis=False,
    hovertemplate=(
        "<b>%{y}</b><br><br>"
        "Lowest: %{customdata[0]} — %{customdata[1]:.1f}%<br>"
        "Highest: %{customdata[2]} — %{customdata[3]:.1f}%<br>"
        "<b>Gap: %{x:.1f} pp</b>"
        "<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_white",
    height=400,
    title=dict(
        text=(
            "<b>Largest Account Ownership Gaps</b><br>"
            "<sup>Difference between the highest and lowest groups within each demographic factor</sup>"
        ),
        x=0.02
    ),
    xaxis_title="Percentage-Point Gap",
    yaxis_title="",
    showlegend=False
)

fig.show()

## RQ1 Key Findings

**Education and income define the largest financial-access gaps.**  
Account ownership ranges from **33.62% among respondents with primary education or less to 82.15% among tertiary-educated respondents**, a **48.53 percentage-point gap**. Income shows the second-largest observed disparity at **31.42 percentage points**. These results identify lower-education and lower-income groups as important financially excluded segments in the descriptive analysis.

# Phase 4 — RQ2: Digital Activation

## Research Question
**Among financial account owners, which groups show the largest differences in digital-payment adoption?**

This phase separates financial access from active digital usage and focuses on the account-owner population.

In [ ]:
# Validate nesting of account ownership and digital-payment usage

account_digital_check = pd.crosstab(
    df["account"],
    df["anydigpayment"],
    margins=True
)

account_digital_check

In [ ]:
# Create account-owner population and calculate activation

account_owners = df[
    df["account"] == 1
].copy()

digital_adoption_account_owners = weighted_percentage(
    account_owners,
    account_owners["anydigpayment"] == 1
)

digital_activation_gap = (
    100 - digital_adoption_account_owners
)

print(
    f"Digital Adoption Among Account Owners: "
    f"{digital_adoption_account_owners:.2f}%"
)

print(
    f"Digital Activation Gap: "
    f"{digital_activation_gap:.2f}%"
)

In [ ]:
# Compare digital-payment adoption across factors

digital_factors = {
    "Internet": "Internet_Status",
    "Education": "Education",
    "Income": "Income_Quintile",
    "Age": "Age_Group",
    "Gender": "Gender",
    "Residence": "Residence",
    "Employment": "Employment"
}

digital_results = {}

for factor, column in digital_factors.items():
    digital_results[factor] = weighted_group_percentage(
        account_owners,
        column,
        "anydigpayment"
    )

digital_gap_rows = []

for factor, table in digital_results.items():
    column = digital_factors[factor]

    lowest = table.loc[
        table["Weighted_Percentage"].idxmin()
    ]

    highest = table.loc[
        table["Weighted_Percentage"].idxmax()
    ]

    digital_gap_rows.append({
        "Factor": factor,
        "Lowest_Group": lowest[column],
        "Lowest_Rate": lowest["Weighted_Percentage"],
        "Highest_Group": highest[column],
        "Highest_Rate": highest["Weighted_Percentage"],
        "Gap_pp": round(
            highest["Weighted_Percentage"]
            - lowest["Weighted_Percentage"],
            2
        )
    })

digital_gap_summary = (
    pd.DataFrame(digital_gap_rows)
    .sort_values("Gap_pp", ascending=False)
    .reset_index(drop=True)
)

digital_gap_summary

In [ ]:
#@title Digital Payment Adoption Gap by Factor

plot_data = (
    digital_gap_summary
    .sort_values("Gap_pp")
)

fig = px.bar(
    plot_data,
    x="Gap_pp",
    y="Factor",
    orientation="h",
    text="Gap_pp",
    custom_data=[
        "Lowest_Group",
        "Lowest_Rate",
        "Highest_Group",
        "Highest_Rate"
    ]
)

fig.update_traces(
    texttemplate="<b>%{text:.1f} pp</b>",
    textposition="outside",
    cliponaxis=False,
    hovertemplate=(
        "<b>%{y}</b><br><br>"
        "Lowest: %{customdata[0]} — %{customdata[1]:.1f}%<br>"
        "Highest: %{customdata[2]} — %{customdata[3]:.1f}%<br>"
        "<b>Gap: %{x:.1f} pp</b>"
        "<extra></extra>"
    )
)

fig.update_layout(
    template="plotly_white",
    height=500,
    title=dict(
        text=(
            "<b>Digital Payment Adoption Gap by Factor</b><br>"
            "<sup>Differences among financial account owners</sup>"
        ),
        x=0.02
    ),
    xaxis_title="Percentage-Point Gap",
    yaxis_title="",
    showlegend=False
)

fig.show()

## RQ2 Key Findings

**Account ownership does not guarantee digital participation.**  
Among account owners, **80.37% use digital payments**, while **19.63% remain digitally inactive**.

**Age and internet usage are key differentiators of digital adoption.**  
Digital-payment adoption ranges from **50.04% among account owners aged 55–64 to 94.33% among those aged 15–24**, a **44.29 percentage-point gap**. Internet users record **87.52% adoption**, compared with **48.29% among non-internet users**, a **39.23 percentage-point gap**.

# Phase 5 — RQ3: Financial Behavior

## Research Question
**How do saving and borrowing behaviors differ across financial-engagement levels?**

This phase examines whether financial participation differs between Financially Excluded, Basic Access, and Digitally Active respondents.

In [ ]:
# Weighted financial behavior segments

behavior_segments = []

for segment in df["Financial_Behavior"].dropna().unique():
    segment_data = df[
        df["Financial_Behavior"] == segment
    ]

    share = (
        segment_data["wgt"].sum()
        / df["wgt"].sum()
        * 100
    )

    behavior_segments.append({
        "Financial_Behavior": segment,
        "Respondents": len(segment_data),
        "Weighted_Percentage": round(share, 2)
    })

behavior_segments = (
    pd.DataFrame(behavior_segments)
    .sort_values("Weighted_Percentage", ascending=False)
    .reset_index(drop=True)
)

behavior_segments

In [ ]:
# Saving and borrowing by financial engagement

saving_by_engagement = weighted_group_percentage(
    df,
    "Financial_Engagement",
    "saved"
)

borrowing_by_engagement = weighted_group_percentage(
    df,
    "Financial_Engagement",
    "borrowed"
)

engagement_behavior = (
    saving_by_engagement[
        ["Financial_Engagement", "Weighted_Percentage"]
    ]
    .rename(
        columns={"Weighted_Percentage": "Saving_Rate"}
    )
    .merge(
        borrowing_by_engagement[
            ["Financial_Engagement", "Weighted_Percentage"]
        ].rename(
            columns={"Weighted_Percentage": "Borrowing_Rate"}
        ),
        on="Financial_Engagement"
    )
)

engagement_behavior

In [ ]:
#@title Financial Behavior by Engagement Level

plot_data = engagement_behavior.melt(
    id_vars="Financial_Engagement",
    value_vars=["Saving_Rate", "Borrowing_Rate"],
    var_name="Behavior",
    value_name="Weighted_Percentage"
)

plot_data["Behavior"] = plot_data["Behavior"].replace({
    "Saving_Rate": "Saved Money",
    "Borrowing_Rate": "Borrowed Money"
})

fig = px.bar(
    plot_data,
    x="Financial_Engagement",
    y="Weighted_Percentage",
    color="Behavior",
    barmode="group",
    text="Weighted_Percentage"
)

fig.update_traces(
    texttemplate="<b>%{text:.1f}%</b>",
    textposition="outside",
    cliponaxis=False
)

fig.update_layout(
    template="plotly_white",
    height=460,
    title=dict(
        text=(
            "<b>Financial Behavior by Engagement Level</b><br>"
            "<sup>Saving and borrowing across levels of financial participation</sup>"
        ),
        x=0.02
    ),
    xaxis=dict(
        title="",
        categoryorder="array",
        categoryarray=[
            "Financially Excluded",
            "Basic Access",
            "Digitally Active"
        ]
    ),
    yaxis=dict(
        title="Weighted Percentage (%)",
        range=[0, 100],
        ticksuffix="%"
    ),
    legend_title=""
)

fig.show()

## RQ3 Key Findings

**Financial engagement differentiates saving more strongly than borrowing.**  
Saving ranges from **36.75% among financially excluded respondents to 77.10% among digitally active respondents**, producing a **40.35 percentage-point gap**. Borrowing ranges from **65.66% to 81.11%**, a smaller **15.45 percentage-point gap**.

**Financial exclusion does not necessarily mean financial inactivity.**  
Borrowing remains relatively widespread among financially excluded respondents, indicating that financial activity and formal financial inclusion are not the same concept.

# Phase 6 — Statistical Validation

Chi-square tests of independence are used to assess sample-level categorical associations. Cramér's V is used to compare relative association strength.

> Survey-weighted percentages remain the primary basis for population-level descriptive interpretation. Statistical association does not imply causation.

In [ ]:
# Statistical libraries and reusable association function

from scipy.stats import chi2_contingency


def categorical_association(data, variable, outcome):
    table = pd.crosstab(
        data[variable],
        data[outcome]
    )

    chi2, p_value, dof, expected = chi2_contingency(
        table
    )

    n = table.to_numpy().sum()
    rows, cols = table.shape

    cramers_v = np.sqrt(
        (chi2 / n)
        / min(rows - 1, cols - 1)
    )

    return {
        "Variable": variable,
        "Chi_Square": round(chi2, 3),
        "Degrees_of_Freedom": dof,
        "P_Value": p_value,
        "Cramers_V": round(cramers_v, 3)
    }

In [ ]:
# RQ1 — Statistical validation for account ownership

rq1_variables = [
    "Education",
    "Income_Quintile",
    "Employment",
    "Age_Group",
    "Residence",
    "Gender"
]

rq1_stats = pd.DataFrame([
    categorical_association(
        df,
        variable,
        "account"
    )
    for variable in rq1_variables
])

rq1_stats["Significant_0.05"] = np.where(
    rq1_stats["P_Value"] < 0.05,
    "Yes",
    "No"
)

rq1_stats = (
    rq1_stats
    .sort_values("Cramers_V", ascending=False)
    .reset_index(drop=True)
)

rq1_stats

In [ ]:
# RQ2 — Statistical validation for digital-payment adoption

rq2_variables = [
    "Age_Group",
    "Internet_Status",
    "Residence",
    "Income_Quintile",
    "Education",
    "Gender",
    "Employment"
]

rq2_stats = pd.DataFrame([
    categorical_association(
        account_owners,
        variable,
        "anydigpayment"
    )
    for variable in rq2_variables
])

rq2_stats["Significant_0.05"] = np.where(
    rq2_stats["P_Value"] < 0.05,
    "Yes",
    "No"
)

rq2_stats = (
    rq2_stats
    .sort_values("Cramers_V", ascending=False)
    .reset_index(drop=True)
)

rq2_stats

In [ ]:
# RQ3 — Statistical validation for financial behavior

rq3_stats = []

for outcome in ["saved", "borrowed"]:
    result = categorical_association(
        df,
        "Financial_Engagement",
        outcome
    )

    result["Outcome"] = (
        "Saving"
        if outcome == "saved"
        else "Borrowing"
    )

    rq3_stats.append(result)

rq3_stats = pd.DataFrame(rq3_stats)

rq3_stats["Significant_0.05"] = np.where(
    rq3_stats["P_Value"] < 0.05,
    "Yes",
    "No"
)

rq3_stats[
    [
        "Outcome",
        "Chi_Square",
        "Degrees_of_Freedom",
        "P_Value",
        "Cramers_V",
        "Significant_0.05"
    ]
]

## Statistical Findings & Interpretation

**RQ1 — Financial Access**  
Education showed the strongest sample-level association with account ownership (**Cramér's V = 0.249, p < 0.001**), followed by income (**V = 0.203, p < 0.001**). These results reinforce the two largest weighted account-ownership gaps.

**RQ2 — Digital Activation**  
Among account owners, age (**V = 0.370, p < 0.001**) and internet usage (**V = 0.354, p < 0.001**) showed the strongest associations with digital-payment adoption. Gender and employment did not show statistically detectable associations at the 5% level.

**RQ3 — Financial Behavior**  
Financial engagement was associated with both saving and borrowing, but the association was considerably stronger for saving (**V = 0.313, p < 0.001**) than borrowing (**V = 0.145, p < 0.001**).

### Overall Statistical Takeaway

The statistical validation generally reinforces the major patterns identified through the weighted descriptive analysis:

**Financial Access → Education & Income**  
**Digital Activation → Age & Internet Usage**  
**Financial Behavior → Stronger differentiation in Saving than Borrowing**

# Phase 7 — Final Synthesis & Recommendations

## Key Insights

**Insight 1: Education and income define the largest financial access gaps**  
Account ownership varied substantially across socioeconomic groups, with a **48.53 percentage-point gap by education** and a **31.42 percentage-point gap by income**. Statistical testing also identified education and income as the strongest demographic associations with account ownership, suggesting that financial access remains particularly uneven across education and income levels.

**Insight 2: Account ownership does not guarantee digital participation**  
Although **80.37% of account owners use digital payments**, **19.63% remain digitally inactive**. This suggests that gaining access to a financial account is only one stage of financial inclusion, and that digital activation should be considered separately from account acquisition.

**Insight 3: Age and internet usage are key differentiators of digital adoption**  
Among account owners, digital-payment adoption showed a **44.29 percentage-point gap across age groups** and a **39.23 percentage-point gap between internet and non-internet users**. These were also the strongest statistical associations with digital-payment adoption, suggesting that digital inclusion efforts may need to address age-related and connectivity-related barriers.

**Insight 4: Financial engagement differentiates saving more strongly than borrowing**  
Saving showed a **40.35 percentage-point gap across financial-engagement levels**, compared with only **15.45 percentage points for borrowing**. Borrowing remained relatively common even among financially excluded respondents, suggesting that financial exclusion does not necessarily mean financial inactivity.

## Recommendations

| Priority | Recommendation | Based On | Suggested Owner |
|---|---|---|---|
| **High** | Prioritize simplified and assisted financial-access initiatives for lower-education and lower-income groups | **Insight 1:** Education and income show the largest financial access gaps | Financial Inclusion / Retail Banking |
| **High** | Track digital activation separately from account acquisition and provide post-opening support for digitally inactive account owners | **Insight 2:** 19.63% of account owners remain digitally inactive | Digital Banking / Customer Engagement |
| **High** | Strengthen assisted digital onboarding and digital-literacy support for older and non-internet users | **Insight 3:** Age and internet usage are key differentiators of digital adoption | Digital Banking / Branch Operations |
| **Medium** | Develop accessible saving initiatives for financially excluded and basic-access groups | **Insight 4:** Saving differs more strongly across financial-engagement levels than borrowing | Deposits / Product Strategy |
| **Low** | Explore broader financial-inclusion measures that combine account ownership, digital activity, saving, and borrowing rather than relying on a single indicator | **Insight 4:** Financial exclusion does not necessarily mean financial inactivity | Financial Inclusion / Consumer Research |

## Assumptions & Limitations

### Assumptions

- The provided survey weight (`wgt`) is treated as the appropriate weighting variable for population-level descriptive estimates.
- Source variable definitions and coding are interpreted according to Global Findex documentation.
- Binary indicators such as `account`, `anydigpayment`, `saved`, and `borrowed` represent reported participation in the corresponding activity.
- The Financial Engagement and Financial Behavior categories are analyst-created simplifications for this project.
- Statistical significance is evaluated at the 5% level.

### Limitations

- The analysis is based on a cross-sectional survey and cannot establish causal relationships.
- The Philippines sample contains 1,000 respondents; some subgroup estimates may therefore rely on relatively small samples.
- Financial behaviors are self-reported and may be affected by recall or response bias.
- Behavioral indicators identify whether an activity occurred but do not capture frequency, value, duration, or intensity.
- Chi-square and Cramér's V are supplementary sample-level measures; the project does not implement a complete complex-survey inference framework.
- Potential confounding factors are not controlled simultaneously. Multivariable modeling would be a useful future extension.

## Conclusion

Financial inclusion in the Philippines extends beyond simply owning a financial account.

The analysis identifies substantial financial-access disparities across education and income groups. Among existing account owners, most use digital payments, but a meaningful activation gap remains, with age and internet usage emerging as the most prominent differentiators of digital adoption.

Financial behavior adds another dimension: saving varies considerably across financial-engagement levels, while borrowing remains widespread even among financially excluded respondents.

Overall, the evidence supports a multidimensional view of financial inclusion:

### **Financial Access → Digital Activation → Financial Engagement**

Different stages may therefore require different interventions, from expanding access among underserved groups to supporting digital activation and encouraging accessible saving participation.

## Project Takeaway

This notebook demonstrates an end-to-end Data Analyst workflow:

**Business Framing → Data Validation → Data Preparation → Survey-Weighted Analysis → Segmentation → Gap Analysis → Statistical Testing → Visualization → Insights → Recommendations**

The detailed GitHub case study can present the executive story, while this notebook serves as the technical evidence supporting the findings.